In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
!pip install numpy matplotlib seaborn scipy
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [3]:
df=pd.read_csv('/content/drive/MyDrive/python_clean_data.csv')
df.columns

Index(['Unnamed: 0', 'Age', 'Attrition', 'BusinessTravel', 'DailyRate',
       'Department', 'DistanceFromHome', 'Education', 'EducationField',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears',
       'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
       'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

##Statistical tests

In [4]:
def chi_2_cramers_v(df, col1, col2):
    contingency_table = pd.crosstab(df[col1], df[col2])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    if p<0.05:
        n = contingency_table.sum().sum()
        phi2 = chi2 / n
        r, k = contingency_table.shape
        cramers_v = np.sqrt(phi2 / min(r - 1, k - 1))
        print(f"Chi-square: {chi2:.2f}")
        print(f"p-value: {p:.2f}.", f" p-value is <0.05; Null hypothesis is rejected; {col1} and {col2} are associated")
        print(f"Cramer's V: {cramers_v:.2f}")
        if cramers_v < 0.1:
          print('Negligible')
        elif 0.1 <= cramers_v < 0.3:
          print('Minor')
        elif 0.3 <= cramers_v < 0.5:
          print('Moderate')
        elif 0.5 <= cramers_v < 0.7:
          print('Strong')
        else:
          print('Very strong')
    else:
        print(f"Chi-square: {chi2:.2f}")
        print(f"p-value: {p:.2f}.", f" p-value is >=0.05; Null hypothesis is not rejected;{col1} and {col2}  are independent")

In [5]:
def get_categories(col):
  col_unique_values = df[col].unique()
  unique_values = col_unique_values.tolist()
  for value in unique_values:
    globals()[value] = df[df[col] == value]
  return unique_values

In [6]:
def anova_or_kruskal_wallis(df, col1, col2):
  from scipy.stats import shapiro, levene, f_oneway, kruskal
  from statsmodels.stats.multicomp import pairwise_tukeyhsd

  print(f"--- Normality Test (Shapiro-Wilk) for {col2} across {col1} groups ---")
  group_data_for_tests = []
  for group_name in df[col1].unique():
    group_data = df[df[col1] == group_name][col2].dropna()
    if len(group_data) >= 3:
      stat_shapiro, p_shapiro = shapiro(group_data)
      print(f"  Group '{group_name}': Shapiro-Wilk W={stat_shapiro:.3f}, p-value={p_shapiro:.3f}")
    else:
      print(f"  Group '{group_name}': Not enough samples ({len(group_data)}) for Shapiro-Wilk test (need at least 3).")
    group_data_for_tests.append(group_data)

  print(f"\n--- Homogeneity of Variances Test (Levene's) for {col2} by {col1} ---")
  use_anova = False
  if len(group_data_for_tests) >= 2 and all(len(g) > 0 for g in group_data_for_tests):
    stat_levene, p_levene = levene(*group_data_for_tests) # Corrected syntax: unpack list of group data
    print(f"  Levene's Statistic: {stat_levene:.3f}")
    print(f"  Levene's p-value: {p_levene:.3f}")

    if p_levene < 0.05:
      print("  Conclusion: p < 0.05. Reject H0; variances are significantly different. Consider Kruskal-Wallis.")
      use_anova = False
    else:
      print("  Conclusion: p >= 0.05. Fail to reject H0; variances are not significantly different. ANOVA assumptions for equal variances are met.")
      use_anova = True
  else:
    print("  Not enough groups or data for Levene's test. Defaulting to non-parametric test.")
    use_anova = False

  print(f"\n--- Hypothesis Testing ---")
  print(f"  Null hypothesis (H₀): The mean/median of {col2} is the same across all {col1} groups.")
  print(f"  Alternative hypothesis (H₁): At least one group has a different mean/median {col2}.")

  if use_anova:
    print("\n  Performing One-Way ANOVA (Parametric Test):")
    stat_anova, p_anova = f_oneway(*group_data_for_tests)
    print(f"    ANOVA F-statistic: {stat_anova:.3f}")
    print(f"    ANOVA p-value: {p_anova:.3f}")
    if p_anova < 0.05:
      print(f"    Conclusion: p < 0.05. Reject H0; there is a significant difference in means of {col2} across {col1} groups.")
      print("\n  Performing Tukey's HSD Post-Hoc Test (for ANOVA):")
      tukey = pairwise_tukeyhsd(
          endog=df[col2],
          groups=df[col1],
          alpha=0.05
      )
      print(tukey)
    else:
      print(f"    Conclusion: p >= 0.05. Fail to reject H0; no significant difference in means of {col2} across {col1} groups.")
  else:
    print("\n  Performing Kruskal-Wallis H-test (Non-Parametric Test):")
    stat_kruskal, p_kruskal = kruskal(*group_data_for_tests)
    print(f"    Kruskal-Wallis H-statistic: {stat_kruskal:.3f}")
    print(f"    Kruskal-Wallis p-value: {p_kruskal:.3f}")
    if p_kruskal < 0.05:
      print(f"    Conclusion: p < 0.05. Reject H0; there is a significant difference in medians of {col2} across {col1} groups.")
    else:
      print(f"    Conclusion: p >= 0.05. Fail to reject H0; no significant difference in medians of {col2} across {col1} groups.")

In [7]:
from scipy.stats import mannwhitneyu
from scipy.stats import shapiro
from scipy.stats import ttest_ind
def t_test_mann_witney(df, col1, col2):
  left = df[df[col1] == True][col2]
  stayed = df[df[col1] == False][col2]

  stat_left, p_left = shapiro(left)
  stat_stayed, p_stayed = shapiro(stayed)

  if p_left > 0.05 and p_stayed > 0.05:
    print('Shapiro test: no evidence against normality as p values are geater that 0.05.')
    print('t-test is applied.')
    stat_t, p_value = ttest_ind(left, stayed, equal_var=False)
    print(f"  t-statistic: {stat_t:.3f}")
    print(f"  p-value: {p_value:.3f}")
    if p_value < 0.05:
      print(f"  Conclusion: p < 0.05. Reject H0; there is a significant difference in the mean of {col2} between {col1} groups.")
    else:
      print(f"  Conclusion: p >= 0.05. Fail to reject H0; no significant difference in the mean of {col2} between {col1} groups.")
  else:
    print('Shapiro test: evidence against normality as p values are less that 0.05.')
    print('Mann-Whitney test is applied.')
    u_stat, p_value = mannwhitneyu(left, stayed, alternative="two-sided")
    print(f"  U-statistic: {u_stat:.3f}")
    print(f"  p-value: {p_value:.3f}")
    if p_value < 0.05:
      print(f"U statistic: {u_stat}")
      print(f"P-value: {p_value}. Null hypothesis is rejected: there is statistical significant dependance between {col1} and {col2}")
      n1 = len(left)
      n2 = len(stayed)
      rank_biserial = 1 - (2 * u_stat) / (n1 * n2)
      if 0.1 <= rank_biserial < 0.3:
        print("Small effect")
      elif 0.3<= rank_biserial < 0.5:
        print("Medium effect")
      elif 0.5<= rank_biserial:
        print("Large effect")
      else:
        print("no effect")
    else:
      print(f"U statistic: {u_stat}")
      print(f"P-value: {p_value}. Null hypothesis is failed to reject: there is no statistical significant dependance between {col1} and {col2}")

##Chi Square

In [8]:
from scipy.stats import chi2_contingency
chi_2_cramers_v(df, 'Attrition', 'Department')

Chi-square: 10.80
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; Attrition and Department are associated
Cramer's V: 0.09
Negligible


In [9]:
chi_2_cramers_v(df, 'Attrition', 'JobRole') ###

Chi-square: 86.19
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; Attrition and JobRole are associated
Cramer's V: 0.24
Minor


In [10]:
chi_2_cramers_v(df, 'Attrition', 'Gender')

Chi-square: 1.12
p-value: 0.29.  p-value is >=0.05; Null hypothesis is not rejected;Attrition and Gender  are independent


In [11]:
chi_2_cramers_v(df, 'Attrition', 'MaritalStatus')###

Chi-square: 46.16
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; Attrition and MaritalStatus are associated
Cramer's V: 0.18
Minor


In [12]:
chi_2_cramers_v(df, 'Attrition', 'Education')

Chi-square: 3.07
p-value: 0.55.  p-value is >=0.05; Null hypothesis is not rejected;Attrition and Education  are independent


In [13]:
chi_2_cramers_v(df, 'Attrition', 'EducationField')

Chi-square: 16.02
p-value: 0.01.  p-value is <0.05; Null hypothesis is rejected; Attrition and EducationField are associated
Cramer's V: 0.10
Minor


In [14]:
chi_2_cramers_v(df, 'Attrition', 'OverTime') ###

Chi-square: 87.56
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; Attrition and OverTime are associated
Cramer's V: 0.24
Minor


In [15]:
chi_2_cramers_v(df, 'MonthlyIncome', 'Attrition')

Chi-square: 1319.02
p-value: 0.71.  p-value is >=0.05; Null hypothesis is not rejected;MonthlyIncome and Attrition  are independent


In [16]:
chi_2_cramers_v(df, 'MonthlyIncome', 'OverTime')

Chi-square: 1341.04
p-value: 0.55.  p-value is >=0.05; Null hypothesis is not rejected;MonthlyIncome and OverTime  are independent


In [17]:
chi_2_cramers_v(df, 'WorkLifeBalance', 'Attrition') ###

Chi-square: 16.33
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; WorkLifeBalance and Attrition are associated
Cramer's V: 0.11
Minor


In [18]:
chi_2_cramers_v(df, 'BusinessTravel', 'Attrition') ###

Chi-square: 24.18
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; BusinessTravel and Attrition are associated
Cramer's V: 0.13
Minor


In [19]:
chi_2_cramers_v(df, 'EnvironmentSatisfaction', 'Attrition') ###

Chi-square: 22.50
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; EnvironmentSatisfaction and Attrition are associated
Cramer's V: 0.12
Minor


In [20]:
chi_2_cramers_v(df, 'RelationshipSatisfaction', 'Attrition')

Chi-square: 5.24
p-value: 0.15.  p-value is >=0.05; Null hypothesis is not rejected;RelationshipSatisfaction and Attrition  are independent


In [21]:
chi_2_cramers_v(df, 'JobSatisfaction', 'Attrition') ###

Chi-square: 17.51
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; JobSatisfaction and Attrition are associated
Cramer's V: 0.11
Minor


In [22]:
chi_2_cramers_v(df, 'JobInvolvement', 'Attrition')###

Chi-square: 28.49
p-value: 0.00.  p-value is <0.05; Null hypothesis is rejected; JobInvolvement and Attrition are associated
Cramer's V: 0.14
Minor


##Anova

In [23]:
anova_or_kruskal_wallis(df, 'Department', 'MonthlyIncome')

--- Normality Test (Shapiro-Wilk) for MonthlyIncome across Department groups ---
  Group 'sales': Shapiro-Wilk W=0.884, p-value=0.000
  Group 'research & development': Shapiro-Wilk W=0.794, p-value=0.000
  Group 'human resources': Shapiro-Wilk W=0.751, p-value=0.000

--- Homogeneity of Variances Test (Levene's) for MonthlyIncome by Department ---
  Levene's Statistic: 3.734
  Levene's p-value: 0.024
  Conclusion: p < 0.05. Reject H0; variances are significantly different. Consider Kruskal-Wallis.

--- Hypothesis Testing ---
  Null hypothesis (H₀): The mean/median of MonthlyIncome is the same across all Department groups.
  Alternative hypothesis (H₁): At least one group has a different mean/median MonthlyIncome.

  Performing Kruskal-Wallis H-test (Non-Parametric Test):
    Kruskal-Wallis H-statistic: 42.619
    Kruskal-Wallis p-value: 0.000
    Conclusion: p < 0.05. Reject H0; there is a significant difference in medians of MonthlyIncome across Department groups.


In [24]:
anova_or_kruskal_wallis(df, 'JobLevel', 'MonthlyIncome')

--- Normality Test (Shapiro-Wilk) for MonthlyIncome across JobLevel groups ---
  Group '2': Shapiro-Wilk W=0.942, p-value=0.000
  Group '1': Shapiro-Wilk W=0.961, p-value=0.000
  Group '3': Shapiro-Wilk W=0.969, p-value=0.000
  Group '4': Shapiro-Wilk W=0.910, p-value=0.000
  Group '5': Shapiro-Wilk W=0.950, p-value=0.008

--- Homogeneity of Variances Test (Levene's) for MonthlyIncome by JobLevel ---
  Levene's Statistic: 70.755
  Levene's p-value: 0.000
  Conclusion: p < 0.05. Reject H0; variances are significantly different. Consider Kruskal-Wallis.

--- Hypothesis Testing ---
  Null hypothesis (H₀): The mean/median of MonthlyIncome is the same across all JobLevel groups.
  Alternative hypothesis (H₁): At least one group has a different mean/median MonthlyIncome.

  Performing Kruskal-Wallis H-test (Non-Parametric Test):
    Kruskal-Wallis H-statistic: 1245.176
    Kruskal-Wallis p-value: 0.000
    Conclusion: p < 0.05. Reject H0; there is a significant difference in medians of Month

##Independent t-Test and Mann-Witney

In [25]:
t_test_mann_witney(df, 'Attrition', 'MonthlyIncome') ###

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 100620.500
  p-value: 0.000
U statistic: 100620.5
P-value: 2.950830917288873e-14. Null hypothesis is rejected: there is statistical significant dependance between Attrition and MonthlyIncome
Medium effect


In [26]:
t_test_mann_witney(df, 'OverTime', 'MonthlyIncome')

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 221735.500
  p-value: 0.733
U statistic: 221735.5
P-value: 0.7327937712299161. Null hypothesis is failed to reject: there is no statistical significant dependance between OverTime and MonthlyIncome


In [27]:
t_test_mann_witney(df, 'Attrition', 'OverTime')

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 190159.500
  p-value: 0.000
U statistic: 190159.5
P-value: 3.985654490549877e-21. Null hypothesis is rejected: there is statistical significant dependance between Attrition and OverTime
no effect


In [28]:
t_test_mann_witney(df, 'Attrition', 'WorkLifeBalance')

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 135709.500
  p-value: 0.046
U statistic: 135709.5
P-value: 0.04647299591978787. Null hypothesis is rejected: there is statistical significant dependance between Attrition and WorkLifeBalance
no effect


In [29]:
t_test_mann_witney(df, 'Attrition', 'YearsAtCompany') ###

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 102582.000
  p-value: 0.000
U statistic: 102582.0
P-value: 2.916191369956416e-13. Null hypothesis is rejected: there is statistical significant dependance between Attrition and YearsAtCompany
Small effect


In [30]:
original_years_at_role = df['YearsInCurrentRole'] ###

years_at_role_for_binning = pd.DataFrame({'YearsInCurrentRole': original_years_at_role})

years_at_role_for_binning['YearsInCurrentRole'] = pd.cut(
    years_at_role_for_binning['YearsInCurrentRole'],
    bins=[0, 3, 6, 9, 12, 15, 18, np.inf], # Use np.inf for the last bin to cover all values >= 30
    labels=['0-3', '3-6', '6-9', '9-12', '12-15', '15-18', '18+'],
    right=False # Ensures intervals are [a, b), matching your original logic
)
df['YearsInCurrentRole'] = years_at_role_for_binning['YearsInCurrentRole']
df.groupby('YearsInCurrentRole')['Attrition'].value_counts(normalize=True)

current_years_in_current_role_state = df['YearsInCurrentRole']
df['YearsInCurrentRole'] = original_years_at_role

t_test_mann_witney(df, 'Attrition', 'YearsInCurrentRole')

df['YearsInCurrentRole'] = current_years_in_current_role_state

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 105214.000
  p-value: 0.000
U statistic: 105214.0
P-value: 4.429559530312778e-12. Null hypothesis is rejected: there is statistical significant dependance between Attrition and YearsInCurrentRole
Small effect


/tmp/ipykernel_953/3235048561.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('YearsInCurrentRole')['Attrition'].value_counts(normalize=True)


In [31]:
original_years_promotion = df['YearsSinceLastPromotion']

years_from_promortion_for_binning = pd.DataFrame({'YearsSinceLastPromotion': original_years_promotion})

years_from_promortion_for_binning['YearsSinceLastPromotion'] = pd.cut(
    years_from_promortion_for_binning['YearsSinceLastPromotion'],
    bins=[0, 3, 6, 9, 12, 15, 18, np.inf], # Use np.inf for the last bin to cover all values >= 30
    labels=['0-3', '3-6', '6-9', '9-12', '12-15', '15-18', '18+'],
    right=False # Ensures intervals are [a, b), matching your original logic
)
df['YearsSinceLastPromotion'] = years_from_promortion_for_binning['YearsSinceLastPromotion']
df.groupby('YearsSinceLastPromotion')['Attrition'].value_counts(normalize=True)

current_years_promotion_state = df['YearsSinceLastPromotion']
df['YearsSinceLastPromotion'] = original_years_promotion

t_test_mann_witney(df, 'Attrition', 'YearsSinceLastPromotion')

df['YearsSinceLastPromotion'] = current_years_promotion_state

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 134374.000
  p-value: 0.041
U statistic: 134374.0
P-value: 0.0411791057847264. Null hypothesis is rejected: there is statistical significant dependance between Attrition and YearsSinceLastPromotion
no effect


/tmp/ipykernel_953/481096765.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('YearsSinceLastPromotion')['Attrition'].value_counts(normalize=True)


In [32]:
t_test_mann_witney(df, 'Attrition', 'TrainingTimesLastYear')

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 134785.500
  p-value: 0.047
U statistic: 134785.5
P-value: 0.04729570730410349. Null hypothesis is rejected: there is statistical significant dependance between Attrition and TrainingTimesLastYear
no effect


In [33]:
t_test_mann_witney(df, 'Attrition', 'YearsAtCompany') ###

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 102582.000
  p-value: 0.000
U statistic: 102582.0
P-value: 2.916191369956416e-13. Null hypothesis is rejected: there is statistical significant dependance between Attrition and YearsAtCompany
Small effect


In [34]:
temp_years_in_current_role = df['YearsInCurrentRole'] ###
df['YearsInCurrentRole'] = original_years_at_role
t_test_mann_witney(df, 'Attrition', 'YearsInCurrentRole')
df['YearsInCurrentRole'] = temp_years_in_current_role

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 105214.000
  p-value: 0.000
U statistic: 105214.0
P-value: 4.429559530312778e-12. Null hypothesis is rejected: there is statistical significant dependance between Attrition and YearsInCurrentRole
Small effect


In [35]:
temp_years_since_last_promotion = df['YearsSinceLastPromotion']
df['YearsSinceLastPromotion'] = original_years_promotion
t_test_mann_witney(df, 'Attrition', 'YearsSinceLastPromotion')
df['YearsSinceLastPromotion'] = temp_years_since_last_promotion

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 134374.000
  p-value: 0.041
U statistic: 134374.0
P-value: 0.0411791057847264. Null hypothesis is rejected: there is statistical significant dependance between Attrition and YearsSinceLastPromotion
no effect


In [36]:
t_test_mann_witney(df, 'Attrition', 'TrainingTimesLastYear')

Shapiro test: evidence against normality as p values are less that 0.05.
Mann-Whitney test is applied.
  U-statistic: 134785.500
  p-value: 0.047
U statistic: 134785.5
P-value: 0.04729570730410349. Null hypothesis is rejected: there is statistical significant dependance between Attrition and TrainingTimesLastYear
no effect
